In [1]:
import json
from pathlib import Path
from collections import defaultdict
import plotly.graph_objects as go

EXCLUDE_CASES = {75, 76, 77, 156, 157, 158, 159}

def load_run(fpath):
    fpath = Path(fpath)
    if not fpath.exists():
        return None
    with open(fpath) as f:
        return [json.loads(l) for l in f]

def get_pred(e):
    # Check various success fields (baseline vs two-stage)
    compiled = e.get('success') or e.get('lean_verification', {}).get('success', False) or e.get('stage2_success', False)
    if not compiled: return 'NC'
    p = str(e.get('prediction', '')).lower()
    if p in ['true', 'yes']: return 'T'
    if p in ['false', 'no']: return 'F'
    return 'Unc'

def get_gt(e):
    g = str(e.get('ground_truth', '')).lower()
    if g in ['true', 'yes']: return 'T'
    if g in ['false', 'no']: return 'F'
    return 'Unc'

def make_gt_pred(gt, pred):
    if pred == 'NC':
        return 'NC'
    return f"{gt}→{pred}"

# Ordered by GT: T→* first, F→* second, Unc→* last (top to bottom)
CATS = ['T→T', 'T→F', 'T→Unc', 'F→T', 'F→F', 'F→Unc', 'Unc→T', 'Unc→F', 'Unc→Unc', 'NC']

# Colors by GT (green=T, red=F, orange=Unc, gray=NC)
NODE_COLOR_MAP = {
    'T→T': 'rgba(39,174,96,1.0)',
    'T→F': 'rgba(39,174,96,0.55)',
    'T→Unc': 'rgba(39,174,96,0.35)',
    'F→T': 'rgba(231,76,60,0.55)',
    'F→F': 'rgba(231,76,60,1.0)',
    'F→Unc': 'rgba(231,76,60,0.35)',
    'Unc→T': 'rgba(243,156,18,0.55)',
    'Unc→F': 'rgba(243,156,18,0.55)',
    'Unc→Unc': 'rgba(243,156,18,1.0)',
    'NC': 'rgba(149,165,166,0.7)',
}

LINK_COLOR_MAP = {
    'T→T': 'rgba(39,174,96,0.4)',
    'T→F': 'rgba(39,174,96,0.25)',
    'T→Unc': 'rgba(39,174,96,0.18)',
    'F→T': 'rgba(231,76,60,0.25)',
    'F→F': 'rgba(231,76,60,0.4)',
    'F→Unc': 'rgba(231,76,60,0.18)',
    'Unc→T': 'rgba(243,156,18,0.25)',
    'Unc→F': 'rgba(243,156,18,0.25)',
    'Unc→Unc': 'rgba(243,156,18,0.4)',
    'NC': 'rgba(149,165,166,0.3)',
}

NODE_COLORS = [NODE_COLOR_MAP[cat] for cat in CATS]

In [2]:
def get_flows(dataset_dir, model_dir, stages, cond_map, base_path='../results'):
    is_folio = 'folio' in dataset_dir
    flows = defaultdict(lambda: defaultdict(int))
    
    for run in [1, 2, 3]:
        data = {}
        for stage, cond in cond_map.items():
            entries = load_run(f'{base_path}/{dataset_dir}/{model_dir}/{cond}_run{run}/results.jsonl')
            if not entries:
                continue
            for e in entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                gt = get_gt(e)
                pred = get_pred(e)
                gt_pred = make_gt_pred(gt, pred)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx][stage] = gt_pred
        
        for case_idx, stages_data in data.items():
            for i in range(len(stages) - 1):
                if stages[i] in stages_data and stages[i+1] in stages_data:
                    src = stages_data[stages[i]]
                    tgt = stages_data[stages[i+1]]
                    flows[(stages[i], stages[i+1])][(src, tgt)] += 1
    return flows

In [3]:
# ADJUST THESE PARAMETERS
NODE_PAD = 5          # Space between nodes (increase for more space)
NODE_THICKNESS = 10   # Node thickness
FONT_SIZE = 11        # Label font size
HEADER_FONT = 11      # Direction header font
STAGE_FONT = 11       # Stage header font
WIDTH = 950           # Plot width
HEIGHT = 400          # Plot height
LEFT_MARGIN = 0      # Left margin
RIGHT_MARGIN = 0     # Right margin
DOMAIN_LEFT = [0.03, 0.47]   # T-direction domain
DOMAIN_RIGHT = [0.53, 0.97]  # F-direction domain

def make_combined_sankey(flows_t, flows_f, stages, title):
    fig = go.Figure()
    
    for side, (flows, domain_x) in enumerate([
        (flows_t, DOMAIN_LEFT),
        (flows_f, DOMAIN_RIGHT)
    ]):
        labels = CATS * len(stages)
        colors = NODE_COLORS * len(stages)
        
        sources, targets, values, link_colors = [], [], [], []
        for i in range(len(stages) - 1):
            for ci, c1 in enumerate(CATS):
                for cj, c2 in enumerate(CATS):
                    v = flows[(stages[i], stages[i+1])].get((c1, c2), 0)
                    if v > 0:
                        sources.append(i * len(CATS) + ci)
                        targets.append((i + 1) * len(CATS) + cj)
                        values.append(v)
                        link_colors.append(LINK_COLOR_MAP[c1])
        
        sankey = go.Sankey(
            node=dict(
                pad=NODE_PAD,
                thickness=NODE_THICKNESS,
                line=dict(color='black', width=0.2),
                label=labels,
                color=colors,
            ),
            link=dict(source=sources, target=targets, value=values, color=link_colors),
            domain=dict(x=domain_x, y=[0.0, 0.90])
        )
        fig.add_trace(sankey)
    
    # Direction headers
    for x_center, dir_label in [((DOMAIN_LEFT[0]+DOMAIN_LEFT[1])/2, 'T-Direction'), 
                                 ((DOMAIN_RIGHT[0]+DOMAIN_RIGHT[1])/2, 'F-Direction')]:
        fig.add_annotation(
            x=x_center, y=1,
            text=f"<b>{dir_label}</b>",
            showarrow=False,
            font=dict(size=HEADER_FONT, family='Arial Black'),
            xanchor='center'
        )
    
    # Stage headers
    for side, (x_start, x_end) in enumerate([DOMAIN_LEFT, DOMAIN_RIGHT]):
        for i, stage in enumerate(stages):
            x = x_start + (i / (len(stages) - 1)) * (x_end - x_start)
            fig.add_annotation(
                x=x, y=0.95,
                text=f"<b>{stage}</b>",
                showarrow=False,
                font=dict(size=STAGE_FONT, family='Arial'),
                xanchor='center'
            )
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=11, family='Arial Black'), x=0.5, y=0.95),
        font=dict(size=FONT_SIZE, family='Arial'),
        width=WIDTH,
        height=HEIGHT,
        margin=dict(l=LEFT_MARGIN, r=RIGHT_MARGIN, t=45, b=10),
    )
    
    return fig

In [4]:
STAGES = ['Baseline', 'Directed', 'Nudged']
T_CONDS = {'Baseline': 'baseline', 'Directed': 'bidir_true', 'Nudged': 'spooky_true'}
F_CONDS = {'Baseline': 'baseline', 'Directed': 'bidir_false', 'Nudged': 'spooky_false'}

# Test with one config first
ds, ds_dir, m, m_dir = 'FOLIO', 'folio', 'GPT-5', 'gpt-5'

flows_t = get_flows(ds_dir, m_dir, STAGES, T_CONDS)
flows_f = get_flows(ds_dir, m_dir, STAGES, F_CONDS)
fig = make_combined_sankey(flows_t, flows_f, STAGES, f'{ds} / {m}')
fig.show()

In [5]:
# Generate all and save
configs = [
    ('FOLIO', 'folio', 'GPT-5', 'gpt-5'),
    ('FOLIO', 'folio', 'DeepSeek', 'deepseek'),
    ('M-LogiEval', 'multilogieval', 'GPT-5', 'gpt-5'),
    ('M-LogiEval', 'multilogieval', 'DeepSeek', 'deepseek'),
]

import os
os.makedirs('../results/prediction_error_fig', exist_ok=True)

for ds, ds_dir, m, m_dir in configs:
    flows_t = get_flows(ds_dir, m_dir, STAGES, T_CONDS)
    flows_f = get_flows(ds_dir, m_dir, STAGES, F_CONDS)
    fig = make_combined_sankey(flows_t, flows_f, STAGES, f'{ds} / {m}')
    fig.write_image(f'../results/prediction_error_fig/sankey_v15_{ds}_{m_dir}.png', scale=2)
    print(f'Saved: sankey_v15_{ds}_{m_dir}.png')

Saved: sankey_v15_FOLIO_gpt-5.png


Saved: sankey_v15_FOLIO_deepseek.png


Saved: sankey_v15_M-LogiEval_gpt-5.png


Saved: sankey_v15_M-LogiEval_deepseek.png


## Baseline vs Two-Stage Sankey

In [6]:
# Baseline vs Two-Stage comparison
def get_flows_twostage(dataset_dir, model, stages, base_path='../results'):
    is_folio = 'folio' in dataset_dir
    flows = defaultdict(lambda: defaultdict(int))
    
    for run in [1, 2, 3]:
        data = {}
        
        # Load baseline (use gpt-5 baseline for both models as reference)
        baseline_entries = load_run(f'{base_path}/{dataset_dir}/gpt-5/baseline_run{run}/results.jsonl')
        if baseline_entries:
            for e in baseline_entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                gt = get_gt(e)
                pred = get_pred(e)
                gt_pred = make_gt_pred(gt, pred)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx]['Baseline'] = gt_pred
        
        # Load two-stage
        twostage_entries = load_run(f'{base_path}/{dataset_dir}/twostage/{model}_run{run}/results.jsonl')
        if twostage_entries:
            for e in twostage_entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                gt = get_gt(e)
                pred = get_pred(e)
                gt_pred = make_gt_pred(gt, pred)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx]['Two-Stage'] = gt_pred
        
        for case_idx, stages_data in data.items():
            for i in range(len(stages) - 1):
                if stages[i] in stages_data and stages[i+1] in stages_data:
                    src = stages_data[stages[i]]
                    tgt = stages_data[stages[i+1]]
                    flows[(stages[i], stages[i+1])][(src, tgt)] += 1
    return flows

def make_single_sankey(flows, stages, title):
    labels = CATS * len(stages)
    colors = NODE_COLORS * len(stages)
    
    sources, targets, values, link_colors = [], [], [], []
    for i in range(len(stages) - 1):
        for ci, c1 in enumerate(CATS):
            for cj, c2 in enumerate(CATS):
                v = flows[(stages[i], stages[i+1])].get((c1, c2), 0)
                if v > 0:
                    sources.append(i * len(CATS) + ci)
                    targets.append((i + 1) * len(CATS) + cj)
                    values.append(v)
                    link_colors.append(LINK_COLOR_MAP[c1])
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=NODE_PAD,
            thickness=NODE_THICKNESS,
            line=dict(color='black', width=0.2),
            label=labels,
            color=colors,
        ),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    ))
    
    # Stage headers
    for i, stage in enumerate(stages):
        x = i / (len(stages) - 1) if len(stages) > 1 else 0.5
        fig.add_annotation(
            x=x, y=1.08,
            text=f"<b>{stage}</b>",
            showarrow=False,
            font=dict(size=STAGE_FONT, family='Arial'),
            xanchor='center'
        )
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=12, family='Arial Black'), x=0.5, y=0.95),
        font=dict(size=FONT_SIZE, family='Arial'),
        width=500,
        height=400,
        margin=dict(l=20, r=20, t=60, b=10),
    )
    
    return fig

# Generate Baseline vs Two-Stage sankeys
STAGES_TS = ['Baseline', 'Two-Stage']
ts_configs = [
    ('FOLIO', 'folio', 'GPT-5', 'gpt-5'),
    ('FOLIO', 'folio', 'DeepSeek-R1', 'deepseek-r1'),
    ('M-LogiEval', 'multilogieval', 'GPT-5', 'gpt-5'),
    ('M-LogiEval', 'multilogieval', 'DeepSeek-R1', 'deepseek-r1'),
]

for ds, ds_dir, m, m_dir in ts_configs:
    flows = get_flows_twostage(ds_dir, m_dir, STAGES_TS)
    fig = make_single_sankey(flows, STAGES_TS, f'{ds} / {m}: Baseline → Two-Stage')
    fig.write_image(f'../results/prediction_error_fig/sankey_v15_twostage_{ds}_{m_dir}.png', scale=2)
    print(f'Saved: sankey_v15_twostage_{ds}_{m_dir}.png')
    fig.show()

Saved: sankey_v15_twostage_FOLIO_gpt-5.png


Saved: sankey_v15_twostage_FOLIO_deepseek-r1.png


Saved: sankey_v15_twostage_M-LogiEval_gpt-5.png


Saved: sankey_v15_twostage_M-LogiEval_deepseek-r1.png


In [7]:
# Combined horizontal layout: all 4 two-stage sankeys
def make_combined_twostage_sankey(configs, stages, base_path='../results'):
    fig = go.Figure()
    
    n_plots = len(configs)
    plot_width = 1.0 / n_plots
    
    for idx, (ds, ds_dir, m, m_dir) in enumerate(configs):
        flows = get_flows_twostage(ds_dir, m_dir, stages, base_path)
        
        labels = CATS * len(stages)
        colors = NODE_COLORS * len(stages)
        
        sources, targets, values, link_colors = [], [], [], []
        for i in range(len(stages) - 1):
            for ci, c1 in enumerate(CATS):
                for cj, c2 in enumerate(CATS):
                    v = flows[(stages[i], stages[i+1])].get((c1, c2), 0)
                    if v > 0:
                        sources.append(i * len(CATS) + ci)
                        targets.append((i + 1) * len(CATS) + cj)
                        values.append(v)
                        link_colors.append(LINK_COLOR_MAP[c1])
        
        x_start = idx * plot_width + 0.02
        x_end = (idx + 1) * plot_width - 0.02
        
        sankey = go.Sankey(
            node=dict(
                pad=NODE_PAD,
                thickness=NODE_THICKNESS,
                line=dict(color='black', width=0.2),
                label=labels,
                color=colors,
            ),
            link=dict(source=sources, target=targets, value=values, color=link_colors),
            domain=dict(x=[x_start, x_end], y=[0.0, 0.85])
        )
        fig.add_trace(sankey)
        
        # Title for each subplot
        fig.add_annotation(
            x=(x_start + x_end) / 2, y=0.98,
            text=f"<b>{ds} / {m}</b>",
            showarrow=False,
            font=dict(size=10, family='Arial Black'),
            xanchor='center'
        )
        
        # Stage headers
        for i, stage in enumerate(stages):
            x = x_start + (i / (len(stages) - 1)) * (x_end - x_start)
            fig.add_annotation(
                x=x, y=0.90,
                text=f"<b>{stage}</b>",
                showarrow=False,
                font=dict(size=9, family='Arial'),
                xanchor='center'
            )
    
    fig.update_layout(
        title=dict(text="Baseline → Two-Stage Comparison", font=dict(size=12, family='Arial Black'), x=0.5, y=0.99),
        font=dict(size=9, family='Arial'),
        width=1200,
        height=400,
        margin=dict(l=10, r=10, t=50, b=10),
    )
    
    return fig

# Generate combined horizontal sankey
ts_configs = [
    ('FOLIO', 'folio', 'GPT-5', 'gpt-5'),
    ('FOLIO', 'folio', 'DeepSeek-R1', 'deepseek-r1'),
    ('M-LogiEval', 'multilogieval', 'GPT-5', 'gpt-5'),
    ('M-LogiEval', 'multilogieval', 'DeepSeek-R1', 'deepseek-r1'),
]

fig = make_combined_twostage_sankey(ts_configs, STAGES_TS)
fig.write_image('../results/prediction_error_fig/sankey_v15_twostage_combined.png', scale=2)
print('Saved: sankey_v15_twostage_combined.png')
fig.show()

Saved: sankey_v15_twostage_combined.png


In [8]:
# 3-Stage Sankey: GT → Baseline → Two-Stage (simple labels)
SIMPLE_CATS = ['T', 'F', 'Unc', 'NC']

SIMPLE_NODE_COLORS = {
    'T': 'rgba(39,174,96,0.9)',      # Green
    'F': 'rgba(231,76,60,0.9)',       # Red
    'Unc': 'rgba(243,156,18,0.9)',    # Orange
    'NC': 'rgba(149,165,166,0.7)',    # Gray
}

SIMPLE_LINK_COLORS = {
    'T': 'rgba(39,174,96,0.35)',
    'F': 'rgba(231,76,60,0.35)',
    'Unc': 'rgba(243,156,18,0.35)',
    'NC': 'rgba(149,165,166,0.25)',
}

def get_simple_pred(e):
    """Get just the prediction (T/F/Unc/NC)"""
    compiled = e.get('success') or e.get('lean_verification', {}).get('success', False) or e.get('stage2_success', False)
    if not compiled: return 'NC'
    p = str(e.get('prediction', '')).lower()
    if p in ['true', 'yes']: return 'T'
    if p in ['false', 'no']: return 'F'
    return 'Unc'

def get_simple_gt(e):
    """Get just the ground truth (T/F/Unc)"""
    g = str(e.get('ground_truth', '')).lower()
    if g in ['true', 'yes']: return 'T'
    if g in ['false', 'no']: return 'F'
    return 'Unc'

def get_flows_gt_baseline_twostage(dataset_dir, ts_model, base_path='../results'):
    """Get flows for GT → Baseline → Two-Stage"""
    is_folio = 'folio' in dataset_dir
    flows = defaultdict(lambda: defaultdict(int))
    stages = ['GT', 'Baseline', 'Two-Stage']
    
    for run in [1, 2, 3]:
        data = {}
        
        # Load baseline (GPT-5) - use for GT and baseline predictions
        baseline_entries = load_run(f'{base_path}/{dataset_dir}/gpt-5/baseline_run{run}/results.jsonl')
        if baseline_entries:
            for e in baseline_entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                gt = get_simple_gt(e)
                pred = get_simple_pred(e)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx]['GT'] = gt
                data[case_idx]['Baseline'] = pred
        
        # Load two-stage
        ts_entries = load_run(f'{base_path}/{dataset_dir}/twostage/{ts_model}_run{run}/results.jsonl')
        if ts_entries:
            for e in ts_entries:
                case_idx = e.get('case_idx')
                if is_folio and case_idx in EXCLUDE_CASES:
                    continue
                pred = get_simple_pred(e)
                if case_idx not in data:
                    data[case_idx] = {}
                data[case_idx]['Two-Stage'] = pred
        
        # Build flows
        for case_idx, stages_data in data.items():
            for i in range(len(stages) - 1):
                if stages[i] in stages_data and stages[i+1] in stages_data:
                    src = stages_data[stages[i]]
                    tgt = stages_data[stages[i+1]]
                    flows[(stages[i], stages[i+1])][(src, tgt)] += 1
    
    return flows, stages

def make_simple_sankey(flows, stages, title):
    # For GT stage, only T/F/Unc (no NC)
    gt_cats = ['T', 'F', 'Unc']
    pred_cats = SIMPLE_CATS  # T, F, Unc, NC
    
    # Build labels: GT uses 3 cats, others use 4
    labels = gt_cats + pred_cats * (len(stages) - 1)
    colors = [SIMPLE_NODE_COLORS[c] for c in gt_cats] + [SIMPLE_NODE_COLORS[c] for c in pred_cats] * (len(stages) - 1)
    
    sources, targets, values, link_colors = [], [], [], []
    
    # GT -> Baseline (stage 0 -> stage 1)
    for ci, c1 in enumerate(gt_cats):
        for cj, c2 in enumerate(pred_cats):
            v = flows[(stages[0], stages[1])].get((c1, c2), 0)
            if v > 0:
                sources.append(ci)  # GT index
                targets.append(len(gt_cats) + cj)  # Baseline index
                values.append(v)
                link_colors.append(SIMPLE_LINK_COLORS[c1])
    
    # Baseline -> Two-Stage (stage 1 -> stage 2)
    for ci, c1 in enumerate(pred_cats):
        for cj, c2 in enumerate(pred_cats):
            v = flows[(stages[1], stages[2])].get((c1, c2), 0)
            if v > 0:
                sources.append(len(gt_cats) + ci)  # Baseline index
                targets.append(len(gt_cats) + len(pred_cats) + cj)  # Two-Stage index
                values.append(v)
                link_colors.append(SIMPLE_LINK_COLORS[c1])
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=NODE_PAD,
            thickness=NODE_THICKNESS,
            line=dict(color='black', width=0.2),
            label=labels,
            color=colors,
        ),
        link=dict(source=sources, target=targets, value=values, color=link_colors),
    ))
    
    # Stage headers
    for i, stage in enumerate(stages):
        x = i / (len(stages) - 1) if len(stages) > 1 else 0.5
        fig.add_annotation(
            x=x, y=1.08,
            text=f"<b>{stage}</b>",
            showarrow=False,
            font=dict(size=STAGE_FONT, family='Arial'),
            xanchor='center'
        )
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=12, family='Arial Black'), x=0.5, y=0.95),
        font=dict(size=FONT_SIZE, family='Arial'),
        width=700,
        height=400,
        margin=dict(l=20, r=20, t=60, b=10),
    )
    
    return fig

# Generate GT → Baseline → Two-Stage sankeys
ts_configs = [
    ('FOLIO', 'folio', 'GPT-5', 'gpt-5'),
    ('FOLIO', 'folio', 'DeepSeek-R1', 'deepseek-r1'),
    ('M-LogiEval', 'multilogieval', 'GPT-5', 'gpt-5'),
    ('M-LogiEval', 'multilogieval', 'DeepSeek-R1', 'deepseek-r1'),
]

for ds, ds_dir, m, m_dir in ts_configs:
    flows, stages = get_flows_gt_baseline_twostage(ds_dir, m_dir)
    fig = make_simple_sankey(flows, stages, f'{ds} / {m}: GT → Baseline → Two-Stage')
    fig.write_image(f'../results/prediction_error_fig/sankey_v16_gt_{ds}_{m_dir}.png', scale=2)
    print(f'Saved: sankey_v16_gt_{ds}_{m_dir}.png')
    fig.show()

Saved: sankey_v16_gt_FOLIO_gpt-5.png


Saved: sankey_v16_gt_FOLIO_deepseek-r1.png


Saved: sankey_v16_gt_M-LogiEval_gpt-5.png


Saved: sankey_v16_gt_M-LogiEval_deepseek-r1.png
